In [1]:
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


In [4]:
def parse_fingering_file(file_path):
    """
    解析单个 fingering 文件，返回包含所有音符信息的列表。
    """
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            # 去除首尾空白字符并按空格分割
            parts = line.strip().split()
            if len(parts) < 8:
                continue  # 跳过格式不完整的行
            note_id = parts[0]
            onset_time = float(parts[1])
            offset_time = float(parts[2])
            spelled_pitch = parts[3]
            onset_velocity = float(parts[4])
            offset_velocity = float(parts[5])
            channel = int(parts[6])
            finger_number = parts[7]
            
            # 处理指法替换（例如 '3_1'）
            if '_' in finger_number:
                finger_number = finger_number.split('_')[0]  # 仅取主要指法
            
            # 转换指法为整数
            try:
                finger_number = int(finger_number)
            except ValueError:
                finger_number = None  # 处理无法转换的指法
            
            # 解析音高和八度
            pitch_name = ''.join([c for c in spelled_pitch if c.isalpha() or c in ['#', 'b']])
            octave = ''.join([c for c in spelled_pitch if c.isdigit()])
            octave = int(octave) if octave else 4  # 默认八度为4
            
            # 确定手部
            hand = 'right' if channel == 0 else 'left'
            
            # 计算音符持续时间
            duration = offset_time - onset_time
            
            data.append({
                'note_id': note_id,
                'onset_time': onset_time,
                'offset_time': offset_time,
                'spelled_pitch': spelled_pitch,
                'pitch_name': pitch_name,
                'octave': octave,
                'duration': duration,
                'hand': hand,
                'finger_number': finger_number
            })
    return data


In [5]:
def load_pig_dataset(fingering_dir):
    """
    加载 PIG Dataset 中所有 fingering 文件，返回一个包含所有音符数据的 DataFrame。
    """
    all_data = []
    for file_name in os.listdir(fingering_dir):
        if file_name.endswith('.txt'):
            file_path = os.path.join(fingering_dir, file_name)
            piece_data = parse_fingering_file(file_path)
            all_data.extend(piece_data)
    df = pd.DataFrame(all_data)
    return df

# 设置 fingering 文件夹路径
fingering_folder = 'PIGdata/FingeringFiles'  # 请根据实际路径调整

# 加载数据
df = load_pig_dataset(fingering_folder)

# 查看数据
print(df.head())


  note_id  onset_time  offset_time spelled_pitch pitch_name  octave  duration  \
0       0    0.004883     0.248048            E4          E       4  0.243165   
1       1    0.004883     0.133302           G#3         G#       3  0.128419   
2       2    0.141114     0.255373            B3          B       3  0.114259   
3       3    0.263185     0.376955            A3          A       3  0.113770   
4       4    0.384768     0.503421           G#3         G#       3  0.118653   

    hand  finger_number  
0  right              1  
1   left             -3  
2   left             -1  
3   left             -2  
4   left             -3  


In [6]:
# 删除指法缺失的音符
df = df.dropna(subset=['finger_number'])

# 确保指法为整数类型
df['finger_number'] = df['finger_number'].astype(int)

# 处理左手指法为正数（可选，根据模型需求）
# 如果模型不需要区分左右手，可以将左手指法取绝对值
# df['finger_number'] = df.apply(lambda row: abs(row['finger_number']) if row['hand'] == 'left' else row['finger_number'], axis=1)

# 查看数据统计
print(df['hand'].value_counts())
print(df['finger_number'].value_counts())


hand
right    54239
left     45801
Name: count, dtype: int64
finger_number
-1    14654
 1    14396
 2    12972
 3    10757
-5    10201
-2     9653
 5     8307
 4     7807
-3     6585
-4     4708
Name: count, dtype: int64


In [7]:
# 初始化LabelEncoder
le_pitch = LabelEncoder()
le_duration = LabelEncoder()
le_hand = LabelEncoder()
le_fingering = LabelEncoder()

# 对类别特征进行标签编码
df['pitch_encoded'] = le_pitch.fit_transform(df['spelled_pitch'])
df['duration_encoded'] = le_duration.fit_transform(df['duration'].astype(str))
df['hand_encoded'] = le_hand.fit_transform(df['hand'])

# 对目标标签进行标签编码
df['fingering_encoded'] = le_fingering.fit_transform(df['finger_number'])

# 特征和标签
X = df[['pitch_encoded', 'duration_encoded', 'hand_encoded']].values
y = df['fingering_encoded'].values

print(f"特征形状: {X.shape}")
print(f"标签形状: {y.shape}")


特征形状: (100040, 3)
标签形状: (100040,)


In [8]:
# 保存LabelEncoder
with open('le_pitch.pkl', 'wb') as f:
    pickle.dump(le_pitch, f)

with open('le_duration.pkl', 'wb') as f:
    pickle.dump(le_duration, f)

with open('le_hand.pkl', 'wb') as f:
    pickle.dump(le_hand, f)

with open('le_fingering.pkl', 'wb') as f:
    pickle.dump(le_fingering, f)


In [9]:
sequence_length = 10  # 使用前10个音符预测第11个音符

def create_sequences(X, y, seq_length):
    X_seq = []
    y_seq = []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y, sequence_length)

print(f"序列特征形状: {X_seq.shape}")  # (样本数, sequence_length, 特征数量)
print(f"序列标签形状: {y_seq.shape}")  # (样本数,)


序列特征形状: (100030, 10, 3)
序列标签形状: (100030,)


In [10]:
# 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)

print(f"训练集样本数: {X_train.shape[0]}")
print(f"验证集样本数: {X_val.shape[0]}")


训练集样本数: 80024
验证集样本数: 20006


In [11]:
# 将序列数据保存为 npy 文件
np.save('X_train.npy', X_train)
np.save('X_val.npy', X_val)
np.save('y_train.npy', y_train)
np.save('y_val.npy', y_val)

## 特征工程

In [12]:
def get_midi_number(spelled_pitch):
    """
    将拼写音高（如 C4, D#5）转换为 MIDI 编号。
    A4 ≈ 440Hz 对应 MIDI 69。
    """
    pitch = spelled_pitch[:-1]  # 音符名称
    octave = int(spelled_pitch[-1])  # 八度
    note_to_semitone = {'C': 0, 'C#': 1, 'Db': 1,
                        'D': 2, 'D#': 3, 'Eb': 3,
                        'E': 4, 'Fb': 4,
                        'F': 5, 'F#': 6, 'Gb': 6,
                        'G': 7, 'G#': 8, 'Ab': 8,
                        'A': 9, 'A#': 10, 'Bb': 10,
                        'B': 11, 'Cb': 11}
    if pitch in note_to_semitone:
        semitone = note_to_semitone[pitch]
    else:
        semitone = 0  # 默认值或根据需要处理
    midi_number = 12 * (octave + 1) + semitone
    return midi_number

# 添加 MIDI 编号特征
df['midi_number'] = df['spelled_pitch'].apply(get_midi_number)

# 更新特征
X = df[['midi_number', 'duration_encoded', 'hand_encoded']].values

# 重新创建序列
X_seq, y_seq = create_sequences(X, y, sequence_length)

print(f"序列特征形状（包含 MIDI 编号）: {X_seq.shape}")
print(f"序列标签形状: {y_seq.shape}")


序列特征形状（包含 MIDI 编号）: (100030, 10, 3)
序列标签形状: (100030,)


## 模型训练

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [15]:
class FingeringDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # 输入特征
        self.y = torch.tensor(y, dtype=torch.long)     # 指法标签

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [16]:
batch_size = 64

train_dataset = FingeringDataset(X_train, y_train)
val_dataset = FingeringDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


In [17]:
class FingeringLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(FingeringLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # 初始化隐藏状态和细胞状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # 前向传播LSTM
        out, _ = self.lstm(x, (h0, c0))  # out: [batch_size, seq_length, hidden_size]

        # 取最后一个时间步的输出
        out = out[:, -1, :]  # [batch_size, hidden_size]

        # 全连接层
        out = self.fc(out)   # [batch_size, num_classes]
        return out


In [18]:
input_size = X_train.shape[2]  # 特征数量
hidden_size = 128
num_layers = 2
num_classes = len(le_fingering.classes_)

model = FingeringLSTM(input_size, hidden_size, num_layers, num_classes)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(model)


FingeringLSTM(
  (lstm): LSTM(3, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)


In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [20]:
num_epochs = 20
best_val_loss = float('inf')
patience = 5
trigger_times = 0
best_model_state = None

for epoch in range(num_epochs):
    # 训练阶段
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 前向传播
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    train_loss /= len(train_dataset)
    train_accuracy = 100 * correct / total

    # 验证阶段
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            val_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    val_loss /= len(val_dataset)
    val_accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}], "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")

    # 早停检查
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict()
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"Trigger Times: {trigger_times}")
        if trigger_times >= patience:
            print("Early stopping!")
            break

# 加载最佳模型
if best_model_state:
    model.load_state_dict(best_model_state)


Epoch [1/20], Train Loss: 2.2538, Train Acc: 14.59%, Val Loss: 2.2534, Val Acc: 14.90%
Epoch [2/20], Train Loss: 2.2519, Train Acc: 14.65%, Val Loss: 2.2532, Val Acc: 14.26%
Epoch [3/20], Train Loss: 2.2505, Train Acc: 14.74%, Val Loss: 2.2525, Val Acc: 15.05%
Epoch [4/20], Train Loss: 2.2492, Train Acc: 14.91%, Val Loss: 2.2507, Val Acc: 15.19%
Epoch [5/20], Train Loss: 2.2499, Train Acc: 14.65%, Val Loss: 2.2528, Val Acc: 14.82%
Trigger Times: 1
Epoch [6/20], Train Loss: 2.2498, Train Acc: 14.89%, Val Loss: 2.2534, Val Acc: 12.76%
Trigger Times: 2
Epoch [7/20], Train Loss: 2.2497, Train Acc: 14.74%, Val Loss: 2.2513, Val Acc: 15.06%
Trigger Times: 3
Epoch [8/20], Train Loss: 2.2493, Train Acc: 14.82%, Val Loss: 2.2522, Val Acc: 15.02%
Trigger Times: 4
Epoch [9/20], Train Loss: 2.2494, Train Acc: 14.78%, Val Loss: 2.2504, Val Acc: 15.08%
Epoch [10/20], Train Loss: 2.2488, Train Acc: 14.68%, Val Loss: 2.2512, Val Acc: 14.35%
Trigger Times: 1
Epoch [11/20], Train Loss: 2.2486, Train Acc